# Subtraction

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0. Native names and tables present the original signatures and declarations.

In [ ]:
from IPython.display import SVG, display
from sympy import Symbol
from examples.algorithms.subtraction_catalog import (
    subtraction_examples, subtraction_example_graph, symbolic_subtraction_steps,
)
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.arithmetic_call import ArithmeticCall
from flagquantum.compiler._component_trace.resource_generalization import OnEachCall
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

examples = subtraction_examples()

def show_signatures(group):
    display(SVG(field_signature_svg([(name, signature, {}) for name, signature, _ in group])))

def call_title(node):
    if isinstance(node, ArithmeticCall):
        widths = ", ".join(map(str, node.widths))
        return f"{node.operation}({widths})"
    if isinstance(node, OnEachCall):
        return f"X on {node.width} wires"
    return str(node)

def show_calls(example):
    graph = subtraction_example_graph(example)
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls, title=call_title))))
    print("Source-declared leaves", graph.leaf_counts)

## `Subtract`
An n-bit subtraction gate.

Implements $U|a\rangle|b\rangle \rightarrow |a\rangle|a-b\rangle$ using $4n - 4$ T gates.

This construction uses the relation `a - b = ~(~a + b)` to turn the operation into addition. This relation is used in
[Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/pdf/2007.07391)
to turn addition into subtraction conditioned on a qubit.

#### Parameters
 - `a_dtype`: Quantum datatype used to represent the integer a.
 - `b_dtype`: Quantum datatype used to represent the integer b. Must be large enough to hold the result in the output register of a - b, or else it simply drops the most significant bits. If not specified, b_dtype is set to a_dtype. 

#### Registers
 - `a`: A a_dtype.bitsize-sized input register (register a above).
 - `b`: A b_dtype.bitsize-sized input/output register (register b above). 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Page 9.


Words are big-endian. For unequal widths the caller provides clean prefix wires and the signed path extends the high bit. The equal-width symbolic composite below retains casts, named operands and call order. Its symbolic steps describe dependencies without emitting gates.

In [ ]:
subtract_examples = examples[:5]

### Example Instances

In [ ]:
subtract_examples[0]

In [ ]:
subtract_examples[1]

In [ ]:
subtract_examples[2]

In [ ]:
subtract_examples[3]

In [ ]:
print("Symbolic signature", subtract_examples[4][1])
for child, words in symbolic_subtraction_steps(Symbol("n")):
    print(child, words)

#### Graphical Signature

In [ ]:
show_signatures(subtract_examples)

### Call Graph

In [ ]:
show_calls(subtract_examples[0])

## `SubtractFrom`
A version of `Subtract` that subtracts the first register from the second in place.

Implements $U|a
angle|b
angle
ightarrow |a
angle|b - a
angle$, essentially equivalent to
the statement `b -= a`.

#### Parameters
 - `dtype`: Quantum datatype used to represent the integers a, b, and b - a. 

#### Registers
 - `a`: A dtype.bitsize-sized input register (register a above).
 - `b`: A dtype.bitsize-sized input/output register (register b above).


The native equal-width kernel complements b, adds a and complements b again. The caller owns the carry wires and classical feedback slots.

In [ ]:
subtract_from_examples = examples[5:]

### Example Instances

In [ ]:
subtract_from_examples[0]

In [ ]:
subtract_from_examples[1]

In [ ]:
subtract_from_examples[2]

#### Graphical Signature

In [ ]:
show_signatures(subtract_from_examples)

### Call Graph

In [ ]:
show_calls(subtract_from_examples[0])

## Execute through append

Run both concrete subtraction directions with ordinary `circuit.append(op)`. The example verifies signed extension, the self-inverse a-b round trip, the b-a result, and clean work on every sampled trajectory. Dedicated tests verify full complex states with a spectator and exhaustive small measurement histories. Emitted gates are counted independently of the declared resource graph.

In [ ]:
from examples.algorithms.register_subtraction import main

main()